# Imports de las librerías

In [17]:
# Imports de las librerías
import sqlite3
from bs4 import BeautifulSoup
import requests
import time
import re

# Conexión a la BD (SQLite como DBMS)

In [18]:
# Conexión a la BD
connection = sqlite3.connect("pengu_books.db")
cursor = connection.cursor()

# FK activadas manualmente
cursor.execute("PRAGMA foreign_keys = ON")
print("Se crea la BD pengu_books")

Se crea la BD pengu_books


# Checking de la página "Books to Scrape"

In [19]:
# Checking rápido si responde la página de Books to Scrape

try:
    # Indicamos la URL 
    url_books_to_scrape = "https://books.toscrape.com/"

    # Se guarda la respuesta del método GET en la variable response
    url_response = requests.get(url_books_to_scrape)

    # Verificación del estado de conexión
    if url_response.status_code == 200:
        print("Conexión exitosa. La página Books to Scrape responde")

except (Exception,KeyboardInterrupt) as error:
    print(f"Hubo un error {error}")

Conexión exitosa. La página Books to Scrape responde


# Variables Globales


In [20]:
URL = "https://books.toscrape.com/"

# Extracción de las categorías

In [21]:
# Se inicia extrayendo inicialmente las categorías de los libros
response = requests.get(URL)
sopa = BeautifulSoup(response.text, "html.parser")

# Capturar la lista del panel lateral
lista_categorias = sopa.find("ul", class_="nav nav-list").find("li").find_all("li")

categorias_data = []
for categoria in lista_categorias:          
    nombre = categoria.text.strip()
    link = URL + categoria.a["href"]
    categorias_data.append((nombre, link))

# Se crea la tabla para las categorías
cursor.execute("""
    CREATE TABLE IF NOT EXISTS categories (
        id   INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT UNIQUE
    )
""")

cursor.executemany(
    "INSERT OR IGNORE INTO categories (name) VALUES (?)",
    [(cat[0],) for cat in categorias_data]
)
connection.commit()

# Caché de mapeo id ↔ nombre
cursor.execute("SELECT id, name FROM categories")
categorias_db = {row[1]: row[0] for row in cursor.fetchall()}

print(f"{len(categorias_db)} categorías guardadas")


50 categorías guardadas


# Extracción de libros